<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Trainable_Tokens_with_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [LoRA Trainable Tokens: Save Memory, Improve Accuracy](https://kaitchup.substack.com/p/lora-trainable-tokens-save-memory)*

This notebook shows how to fine-tune an LLM with LoRA while training new tokens. It supports LoRA and QLoRA with trainable tokens for the token embeddings and language modeling head.



#Installation

In [ ]:
!pip install --upgrade transformers bitsandbytes peft accelerate datasets trl flash_attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 60.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.8/410.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.7/335.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.1 MB/s eta 0:00:00


# Fine-Tuning Code

A unique function to perform QLoRA/LoRA fine-tuning, with trainable tokens or full re-training of the embeddings:

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)

compute_dtype = torch.bfloat16
attn_implementation = 'flash_attention_2'

def fine_tune(model_name, batch_size=1, gradient_accumulation_steps=32, LoRA=False, QLoRA=False, full_retrain_vocab=True, trainable_tokens=None):

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = "<|finetune_right_pad_id|>"
  tokenizer.pad_token_id = 128004
  tokenizer.padding_side = 'right'

  tokenizer_name_chat_template = "meta-llama/Llama-3.1-8B-Instruct"
  tokenizer_chat = AutoTokenizer.from_pretrained(tokenizer_name_chat_template)

  ds_train = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:20000]")
  ds_test = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft[:1000]")

  def process(row):
      prompt_messages = tokenizer_chat.apply_chat_template(row["messages"], tokenize=False)

      row["text"] = prompt_messages
      return row

  ds_train = ds_train.map(
      process,
      num_proc= multiprocessing.cpu_count(),
      load_from_cache_file=False,
  )

  ds_test = ds_test.map(
      process,
      num_proc= multiprocessing.cpu_count(),
      load_from_cache_file=False,
  )
  ds_train = ds_train.remove_columns(["messages", "prompt"])
  ds_test = ds_test.remove_columns(["messages", "prompt"])

  if QLoRA:
    bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
              model_name, quantization_config=bnb_config, device_map={"": 0}, attn_implementation=attn_implementation
    )
    model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={'use_reentrant':True})
  else:
    model = AutoModelForCausalLM.from_pretrained(
              model_name, device_map={"": 0}, torch_dtype=compute_dtype, attn_implementation=attn_implementation
    )
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':True})




  if LoRA or QLoRA:
    peft_config = LoraConfig(
            lora_alpha=16,
            lora_dropout=0.05,
            r=16,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"],
            modules_to_save = ["embed_tokens", "lm_head"] if full_retrain_vocab else None,
            trainable_token_indices = trainable_tokens,
    )
  else:
      peft_config = None

  if LoRA:
    output_dir = "./LoRA/"
  elif QLoRA:
    output_dir = "./QLoRA/"
  else:
    output_dir = "./FFT/"

  training_arguments = SFTConfig(
          output_dir=output_dir,
          eval_strategy="steps",
          do_eval=True,
          optim="adamw_8bit",
          per_device_train_batch_size=batch_size,
          gradient_accumulation_steps=gradient_accumulation_steps,
          per_device_eval_batch_size=batch_size,
          log_level="debug",
          save_strategy="epoch",
          logging_steps=25,
          learning_rate=1e-5,
          bf16 = True,
          eval_steps=25,
          num_train_epochs=1,
          warmup_ratio=0.1,
          lr_scheduler_type="linear",
          dataset_text_field="text",
          max_seq_length=512,
          report_to="none",
  )

  trainer = SFTTrainer(
          model=model,
          train_dataset=ds_train,
          eval_dataset=ds_test,
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
  )

  #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

  gpu_stats = torch.cuda.get_device_properties(0)
  start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
  print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
  print(f"{start_gpu_memory} GB of memory reserved.")

  trainer_ = trainer.train()


  used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
  used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
  used_percentage = round(used_memory         /max_memory*100, 3)
  trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
  print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
  print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
  print(f"Peak reserved memory = {used_memory} GB.")
  print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
  print(f"Peak reserved memory % of max memory = {used_percentage} %.")
  print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
  print("-----")
  #----

# Trainable Special Tokens from the Chat Template

```
   {
      "id": 128006,
      "content": "<|start_header_id|>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 128007,
      "content": "<|end_header_id|>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 128008,
      "content": "<|eom_id|>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 128009,
      "content": "<|eot_id|>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
```



# Trainable Tokens for The Token Embeddings and LM Head

In [ ]:
fine_tune("meta-llama/Llama-3.2-3B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens={'embed_tokens':[128006, 128007, 128008, 128009], 'lm_head':[128006, 128007, 128008, 128009]})

Map (num_proc=12):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Converting train dataset to ChatML:   0%|          | 0/20000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 625
  Number of trainable parameters = 24,338,432
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GPU = NVIDIA L4. Max memory = 22.161 GB.
6.812 GB of memory reserved.


Step,Training Loss,Validation Loss
25,2.082900,2.077911
50,2.034500,1.999311
75,1.899100,1.823312
100,1.738100,1.662847
125,1.594800,1.565778
150,1.511200,1.514756
175,1.494600,1.480421
200,1.496600,1.458785
225,1.447200,1.443816
250,1.414400,1.432490


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11648.5425 seconds used for training.
194.14 minutes used for training.
Peak reserved memory = 8.766 GB.
Peak reserved memory for training = 1.954 GB.
Peak reserved memory % of max memory = 39.556 %.
Peak reserved memory for training % of max memory = 8.817 %.
-----


# Trainable Tokens Only for The Token Embeddings

In [ ]:
fine_tune("meta-llama/Llama-3.2-3B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens={'embed_tokens':[128006, 128007, 128008, 128009]})

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

GPU = NVIDIA L4. Max memory = 22.161 GB.
6.812 GB of memory reserved.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
25,2.083600,2.079562
50,2.039600,2.007946
75,1.913100,1.843030
100,1.761100,1.688903
125,1.622100,1.598186
150,1.546300,1.553361
175,1.532800,1.518899
200,1.530700,1.492068
225,1.476000,1.471389
250,1.439100,1.455431


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11546.934 seconds used for training.
192.45 minutes used for training.
Peak reserved memory = 8.766 GB.
Peak reserved memory for training = 1.954 GB.
Peak reserved memory % of max memory = 39.556 %.
Peak reserved memory for training % of max memory = 8.817 %.
-----


# Full Retraining of the Token Embeddings and LM Head

In [ ]:
fine_tune("meta-llama/Llama-3.2-3B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=True, trainable_tokens=None)

Map (num_proc=12):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 625
  Number of trainable parameters = 812,318,720
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GPU = NVIDIA L4. Max memory = 22.161 GB.
7.545 GB of memory reserved.


Step,Training Loss,Validation Loss
25,2.084900,2.084275
50,2.044900,2.008957
75,1.903700,1.818273
100,1.726300,1.645280
125,1.575900,1.543401
150,1.487600,1.490171
175,1.472000,1.459314
200,1.478300,1.442241
225,1.432400,1.430138
250,1.401000,1.420192


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11446.6444 seconds used for training.
190.78 minutes used for training.
Peak reserved memory = 12.475 GB.
Peak reserved memory for training = 4.93 GB.
Peak reserved memory % of max memory = 56.293 %.
Peak reserved memory for training % of max memory = 22.246 %.
-----


# Standard LoRA with Frozen Embeddings

In [ ]:
fine_tune("meta-llama/Llama-3.2-3B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens=None)

Map (num_proc=12):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 625
  Number of trainable parameters = 24,313,856
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GPU = NVIDIA L4. Max memory = 22.161 GB.
6.811 GB of memory reserved.


Step,Training Loss,Validation Loss
25,2.086100,2.088994
50,2.060800,2.044523
75,1.962400,1.907369
100,1.832500,1.775008
125,1.710200,1.671503
150,1.620000,1.635396
175,1.619800,1.616680
200,1.629600,1.604209
225,1.588100,1.594141
250,1.565400,1.585501


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

10803.0602 seconds used for training.
180.05 minutes used for training.
Peak reserved memory = 7.521 GB.
Peak reserved memory for training = 0.71 GB.
Peak reserved memory % of max memory = 33.938 %.
Peak reserved memory for training % of max memory = 3.204 %.
-----


# Same but with Llama 3.1 8B

In [ ]:
fine_tune("meta-llama/Llama-3.1-8B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens={'embed_tokens':[128006, 128007, 128008, 128009], 'lm_head':[128006, 128007, 128008, 128009]})

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
15.117 GB of memory reserved.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
25,2.026700,2.018397
50,1.964100,1.935077
75,1.865400,1.821848
100,1.747000,1.664987
125,1.580400,1.532543
150,1.452400,1.442516
175,1.413300,1.396944
200,1.413200,1.368991
225,1.352900,1.350407
250,1.321900,1.342116


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

12229.0841 seconds used for training.
203.82 minutes used for training.
Peak reserved memory = 18.678 GB.
Peak reserved memory for training = 3.561 GB.
Peak reserved memory % of max memory = 47.218 %.
Peak reserved memory for training % of max memory = 9.002 %.
-----


In [ ]:
fine_tune("meta-llama/Llama-3.1-8B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens={'embed_tokens':[128006, 128007, 128008, 128009]})

Map (num_proc=12):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 625
  Number of trainable parameters = 41,959,424
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
15.117 GB of memory reserved.


Step,Training Loss,Validation Loss
25,2.027000,2.018490
50,1.971700,1.953394
75,1.891300,1.863033
100,1.809600,1.765459
125,1.709400,1.682765
150,1.623500,1.631032
175,1.602500,1.584144
200,1.597400,1.558348
225,1.529400,1.534508
250,1.498200,1.513493


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11166.4753 seconds used for training.
186.11 minutes used for training.
Peak reserved memory = 16.721 GB.
Peak reserved memory for training = 1.604 GB.
Peak reserved memory % of max memory = 42.271 %.
Peak reserved memory for training % of max memory = 4.055 %.
-----


In [ ]:
fine_tune("meta-llama/Llama-3.1-8B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=True, trainable_tokens=None)

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
17.074 GB of memory reserved.


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
25,2.026200,2.014361
50,1.964000,1.950299
75,1.871200,1.801763
100,1.700000,1.606958
125,1.559100,1.544479
150,1.486800,1.474585
175,1.439000,1.419900
200,1.423500,1.365935
225,1.349700,1.346635
250,1.316600,1.330740


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: prompt_id, text. If prompt_id, text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11410.2612 seconds used for training.
190.17 minutes used for training.
Peak reserved memory = 23.508 GB.
Peak reserved memory for training = 6.434 GB.
Peak reserved memory % of max memory = 59.428 %.
Peak reserved memory for training % of max memory = 16.265 %.
-----


In [ ]:
fine_tune("meta-llama/Llama-3.1-8B", batch_size=1, gradient_accumulation_steps=32, LoRA=True, QLoRA=False, full_retrain_vocab=False, trainable_tokens=None)

Map (num_proc=12):   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Currently training with a batch size of: 1
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 20,000
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 32
  Total optimization steps = 625
  Number of trainable parameters = 41,943,040
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
15.117 GB of memory reserved.


Step,Training Loss,Validation Loss
25,2.034100,2.040828
50,2.002700,1.964573
75,1.835200,1.738117
100,1.638200,1.575471
125,1.534600,1.526679
150,1.485800,1.505112
175,1.491500,1.489559
200,1.508700,1.477761
225,1.460500,1.468789
250,1.440400,1.460178


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text, prompt_id. If text, prompt_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 1000
  Batch size = 1
T

11103.6948 seconds used for training.
185.06 minutes used for training.
Peak reserved memory = 16.656 GB.
Peak reserved memory for training = 1.539 GB.
Peak reserved memory % of max memory = 42.106 %.
Peak reserved memory for training % of max memory = 3.891 %.
-----
